# Phase 4 — ML Ranking Model

Uses persisted Phase 2 NSE market history and Phase 3 targets to train a time-aware model and rank stocks using the latest available trading date.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from src.data.providers.nse_market import load_market_data

ROOT = Path(__file__).resolve().parents[1]
TARGET_PATH = ROOT / "data/processed/targets/phase3_target.parquet"
OUT_DIR = ROOT / "data/processed/model"
OUT_DIR.mkdir(parents=True, exist_ok=True)

market = load_market_data().copy()
target = pd.read_parquet(TARGET_PATH)
market["trade_date"] = pd.to_datetime(market["trade_date"])
target["decision_date"] = pd.to_datetime(target["decision_date"])
        market = market.sort_values(["isin", "trade_date"])
        
        if target.empty:
            raise ValueError("Phase 3 target dataset is empty.")
        
        print("Market rows :", f"{len(market):,}")
        print("Target rows :", f"{len(target):,}")
        print("Latest date :", market["trade_date"].max().date())

In [ ]:
g = market.groupby("isin")
market["ret_1d"] = g["close"].pct_change()
market["ret_5d"] = g["close"].pct_change(5)
market["ret_20d"] = g["close"].pct_change(20)
market["ret_60d"] = g["close"].pct_change(60)
market["ret_126d"] = g["close"].pct_change(126)
market["ret_252d"] = g["close"].pct_change(252)
market["vol_20d"] = g["ret_1d"].transform(lambda x: x.rolling(20).std())
market["vol_60d"] = g["ret_1d"].transform(lambda x: x.rolling(60).std())
market["vol_126d"] = g["ret_1d"].transform(lambda x: x.rolling(126).std())
market["volume_ratio_20d"] = market["volume"] / g["volume"].transform(lambda x: x.rolling(20).mean())
market["turnover_ratio_20d"] = market["turnover"] / g["turnover"].transform(lambda x: x.rolling(20).mean())
market["price_vs_20d"] = market["close"] / g["close"].transform(lambda x: x.rolling(20).mean()) - 1
market["price_vs_60d"] = market["close"] / g["close"].transform(lambda x: x.rolling(60).mean()) - 1

FEATURES = [
    "ret_1d", "ret_5d", "ret_20d", "ret_60d", "ret_126d", "ret_252d",
    "vol_20d", "vol_60d", "vol_126d",
    "volume_ratio_20d", "turnover_ratio_20d",
    "price_vs_20d", "price_vs_60d"
]

features = market[["isin", "trade_date"] + FEATURES].rename(columns={"trade_date": "decision_date"})
model_df = target.merge(features, on=["isin", "decision_date"], how="inner", validate="one_to_one")
model_df = model_df.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURES + ["target_excess_return_252d"])

print("Training rows:", f"{len(model_df):,}")
print("Companies    :", f"{model_df['isin'].nunique():,}")
print("Decision dates:", f"{model_df['decision_date'].nunique():,}")

In [ ]:
dates = np.sort(model_df["decision_date"].unique())
split = dates[int(len(dates) * 0.80)]
train = model_df[model_df["decision_date"] < split]
valid = model_df[model_df["decision_date"] >= split]

model = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_iter=250,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

model.fit(train[FEATURES], train["target_excess_return_252d"])
pred_valid = model.predict(valid[FEATURES])
mae = mean_absolute_error(valid["target_excess_return_252d"], pred_valid)
r2 = r2_score(valid["target_excess_return_252d"], pred_valid)

eval_df = valid[["decision_date", "target_excess_return_252d"]].copy()
eval_df["prediction"] = pred_valid
daily_ic = eval_df.groupby("decision_date").apply(
    lambda x: x["prediction"].corr(x["target_excess_return_252d"]),
    include_groups=False
).dropna()

print("=== MODEL VALIDATION ===")
print("Train rows:", f"{len(train):,}")
print("Valid rows:", f"{len(valid):,}")
print("Split date :", pd.Timestamp(split).date())
print("MAE        :", round(mae, 6))
print("R2         :", round(r2, 6))
print("Mean Daily IC:", round(daily_ic.mean(), 6))

In [ ]:
model.fit(model_df[FEATURES], model_df["target_excess_return_252d"])

latest_date = market["trade_date"].max()
latest = market[(market["trade_date"] == latest_date) & (market["series"] == "EQ")].copy()
latest = latest.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURES)
latest["predicted_excess_return_252d"] = model.predict(latest[FEATURES])
latest = latest.sort_values("predicted_excess_return_252d", ascending=False)

ranking = latest[[
    "isin", "nse_symbol", "instrument_name", "close",
    "predicted_excess_return_252d"
]].copy()
ranking["rank"] = np.arange(1, len(ranking) + 1)
ranking = ranking[[
    "rank", "isin", "nse_symbol", "instrument_name", "close",
    "predicted_excess_return_252d"
]]

ranking.to_parquet(OUT_DIR / "latest_stock_ranking.parquet", index=False)
model_df.to_parquet(OUT_DIR / "training_dataset.parquet", index=False)

metadata = {
    "model": "HistGradientBoostingRegressor",
    "training_start": str(model_df["decision_date"].min().date()),
    "training_end": str(model_df["decision_date"].max().date()),
    "inference_date": str(latest_date.date()),
    "target": "target_excess_return_252d",
    "features": FEATURES,
    "validation_mae": float(mae),
    "validation_r2": float(r2),
    "mean_daily_information_coefficient": float(daily_ic.mean()),
    "rows_scored": int(len(ranking))
}

(OUT_DIR / "phase4_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
        
        print("=== CURRENT MODEL RANKING ===")
        print("Inference date:", latest_date.date())
        display(ranking.head(25))
        print("\nSaved:")
        print(OUT_DIR / "latest_stock_ranking.parquet")
        print(OUT_DIR / "training_dataset.parquet")
        print(OUT_DIR / "phase4_metadata.json")

## Phase 4 Completion

The model uses historical point-in-time features, time-aware validation, and all available historical targets before ranking stocks on the latest NSE trading date.